### Input data into notebook

Data is read from the CSV file with spark and ingested into the datastream 

In [0]:
# Read Food Inspections CSV with Spark
VOLUME_CSV_PATH = "/Volumes/students_data/team4-chicago/chicago-restaurants"
file_name = "Food_Inspections.csv"
full_path = f"{VOLUME_CSV_PATH}/{file_name}"

df = spark.read.csv(full_path, header=True, inferSchema=True)

print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")
df.printSchema()
display(df.limit(20))

In [0]:
# Rename columns to remove spaces and special characters for Delta compatibility
import re

df_clean = df
for col_name in df.columns:
    new_name = re.sub(r'[^a-zA-Z0-9_]', '_', col_name).strip('_')
    df_clean = df_clean.withColumnRenamed(col_name, new_name)

# Write raw data to a bronze Delta table so downstream notebooks can read it
df_clean.write.mode("overwrite").saveAsTable("students_data.`team4-chicago`.food_inspections_bronze")

print("Bronze table written: students_data.team4-chicago.food_inspections_bronze")
print(f"Rows: {spark.table('students_data.`team4-chicago`.food_inspections_bronze').count():,}")